In [1]:
# ── Celda 1: cargar el DataFrame normalizado usando el propio pipeline ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent          # el notebook vive en analysis/, el repo es su padre
sys.path.insert(0, str(ROOT))

import yaml
from src.ingest import load
from src.normalize import normalize

config = yaml.safe_load((ROOT / "config.yaml").read_text())
records = load(ROOT / "data" / "raw" / "alerts_combined.json")
df = normalize(records, config)

print(df.shape)        # esperado: (458, ~20 columnas)
df.head(3)

(458, 17)


,dt,channel,source,service,service_original,grupo_criticidad,condition,policy,priority,incidents_raw,threshold,tipo_regla,direccion,ventana_eval,error_type,error_message,procesador
0,2025-05-26 15:06:22-06:00,sre,New Relic,Wiki,Wiki,web,Parco APIs status - locations failed,Up_Satatus_Parco,critical,1.0,1+ locations,estatica,None,None,,,
1,2025-05-26 16:07:41-06:00,sre,New Relic,Orchestrator,Orchestrator,nucleo,high request count with status 500,Parco2.0 strict,critical,2.0,>60/5min,estatica,sobre,5min,,,
2,2025-05-26 16:12:41-06:00,sre,New Relic,Orchestrator,Orchestrator,nucleo,High Application Error percentage,Golden Signals,critical,1.0,baseline/10min,anomalia,None,10min,,,


In [2]:
# ── Celda 2: Verificación 1 — la regla de relojes está viva en el código ──
print(df.groupby("channel")["dt"].agg(["min", "max", "count"]), "\n")

sre = df.loc[df["channel"] == "sre", "dt"]
cx  = df.loc[df["channel"] == "monitoring-ops-cx", "dt"]

assert str(sre.min().date()) == "2025-05-26", f"sre min inesperado: {sre.min()}"
assert str(sre.max().date()) == "2025-06-11", f"sre max inesperado: {sre.max()}"
assert str(cx.min().date())  == "2026-03-24", f"cx min inesperado: {cx.min()}"
assert str(cx.max().date())  == "2026-03-27", f"cx max inesperado: {cx.max()}"
assert len(sre) == 363 and len(cx) == 95

print("✓ Relojes correctos: sre vive en may-jun 2025, cx vive en mar 2026")

                                        min                              max  \
channel                                                                        
monitoring-ops-cx 2026-03-24 12:54:07-06:00        2026-03-27 16:21:20-06:00   
sre               2025-05-26 15:06:22-06:00 2025-06-11 07:09:39.517959-06:00   

                   count  
channel                   
monitoring-ops-cx     95  
sre                  363   

✓ Relojes correctos: sre vive en may-jun 2025, cx vive en mar 2026


In [3]:
# ── Celda 3: Verificación 2 — grupos de criticidad y normalización de services ──
print(df["grupo_criticidad"].value_counts(), "\n")

# Esperado (de las pivotes del EDA):
#   pagos    171   (Hairs 124 + Wallet_2.0 31 + Transaction query 12 + Chargehound 2 + PayPal 1 + Invoice 1)
#   nucleo   166   (Orchestrator 104 + Carts 28 + Users 26 + Access 6 + Peajero 2)
#   codename  73   (tesseract 29 + Cerberus 20 + Princess 12* + Kraken 5 + demo2 4 + Gigante 3)
#   infra     30   (11+7+2+2 instancias EC2 + 6+2 new-parco-instance)
#   web       11   (Wiki 9 + Wordpress 1 + Web Page Parco 1)
#   datos      7   (data-team)
#   * Princess 12 = 11 + 1 del typo Princes ya unificado

infra = df[df["grupo_criticidad"] == "infra"]
assert len(infra) == 30, f"infra esperaba 30, salió {len(infra)}"
assert (infra["service"] == "infra").all()
print(infra["service_original"].value_counts(), "\n")   # los 6 hosts originales conservados

assert (df["service"] == "Princess").sum() == 12, "Princess+Princes debían unificar en 12"
assert "Princes" not in set(df["service"]), "el typo Princes sobrevivió a la normalización"
assert df["grupo_criticidad"].value_counts().sum() == 458   # ley de conservación

print("✓ Normalización correcta: typo unificado, infra agrupada con original conservado")

grupo_criticidad
pagos       171
nucleo      166
codename     73
infra        30
web          11
datos         7
Name: count, dtype: int64 

service_original
i-058689de5ec046291     11
i-0d26dd24e2a69bff0      7
new-parco-instance-1     6
new-parco-instance-5     2
i-04acbab68c9357917      2
i-0d568d70a0847d5b6      2
Name: count, dtype: int64 

✓ Normalización correcta: typo unificado, infra agrupada con original conservado
